In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm  # 로딩바 라이브러리

In [2]:
import pandas as pd

# 1. 데이터 불러오기
mos_w_df = pd.read_csv('data\Merge_mos_w_data.csv')
river_df = pd.read_csv('data\River_data.csv')

In [3]:
# 2. 날짜 컬럼(tm) 형식 및 중복 체크
# 두 데이터의 tm 컬럼을 문자열로 통일
mos_w_df['tm'] = mos_w_df['tm'].astype(str)
river_df['tm'] = river_df['tm'].astype(str)

In [4]:

# [중복 제거] 날짜별로 데이터가 여러 개 있다면 평균값으로 압축
if mos_w_df['tm'].duplicated().any():
    print("⚠️ mos_w_df에서 날짜 중복 발견! 중복을 제거합니다.")
    mos_w_df = mos_w_df.groupby('tm').mean(numeric_only=True).reset_index()

if river_df['tm'].duplicated().any():
    print("⚠️ river_df에서 날짜 중복 발견! 중복을 제거합니다.")
    river_df = river_df.groupby('tm').mean(numeric_only=True).reset_index()

In [5]:
# 3. [River_data 결측치 처리] 
# 시계열 특성을 고려하여 선형 보간법(Linear Interpolation) 사용
# 앞뒤 데이터를 이어주는 방식이라 수온/수질 데이터에 가장 적합합니다.
river_df_clean = river_df.interpolate(method='linear', limit_direction='both')

# 혹시 보간법으로도 안 채워진 곳이 있다면 전체 평균으로 보충
river_df_clean = river_df_clean.fillna(river_df_clean.mean(numeric_only=True))

In [6]:
# 4. 최종 병합 (Merge_mos_w_data가 왼쪽으로)
# 'tm' 컬럼을 기준으로 교집합(inner) 병합 수행
final_dataset = pd.merge(mos_w_df, river_df_clean, on='tm', how='inner')

# 5. 결과 확인 및 저장
print("\n--- [최종 데이터셋] 병합 결과 ---")
print(f"최종 행 개수: {len(final_dataset)}행")
print(f"결측치 존재 여부: {final_dataset.isnull().sum().sum()}개")

# 한글 깨짐 방지를 위해 utf-8-sig로 저장
final_dataset.to_csv('Seoul_Mosquito_Final_Dataset.csv', index=False, encoding='utf-8-sig')
display(final_dataset.head())


--- [최종 데이터셋] 병합 결과 ---
최종 행 개수: 1093행
결측치 존재 여부: 0개


,tm,mosquito_value_house,mosquito_value_park,STN,WS_AVG,WR_DAY,WD_MAX,WS_MAX,WS_MAX_TM,WD_INS,...,TE_15,TE_30,TE_50,날짜,수온,pH,용존산소(㎎/L),총질소(㎎/L),총인(㎎/L),총유기탄소
0,20200501,32.6,47.0,108,2.7,2328,29,5.5,1726,29,...,12.6,12.8,13.4,2020-05-01,21.298611,7.298611,6.229167,8.086181,0.116528,6.862500
1,20200502,36.6,52.7,108,2.3,1985,23,4.6,111,29,...,12.7,12.8,13.4,2020-05-02,21.418056,7.183333,5.615278,7.697153,0.201667,5.084722
2,20200503,43.0,62.0,108,2.0,1705,16,3.9,1242,14,...,12.9,12.9,13.4,2020-05-03,22.005556,7.166667,5.736111,7.444375,0.200139,5.004167
3,20200504,43.2,62.3,108,2.7,2310,27,4.6,1247,27,...,13.0,12.9,13.4,2020-05-04,23.216667,7.169444,5.648611,7.402917,0.144861,5.172348
4,20200505,46.7,67.3,108,1.9,1601,25,5.3,1354,23,...,13.2,12.9,13.4,2020-05-05,20.429167,7.102778,5.350000,7.886250,0.209861,5.837500


In [7]:
final_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1093 entries, 0 to 1092
Data columns (total 65 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   tm                    1093 non-null   object 
 1   mosquito_value_house  1093 non-null   float64
 2   mosquito_value_park   1093 non-null   float64
 3   STN                   1093 non-null   int64  
 4   WS_AVG                1093 non-null   float64
 5   WR_DAY                1093 non-null   int64  
 6   WD_MAX                1093 non-null   int64  
 7   WS_MAX                1093 non-null   float64
 8   WS_MAX_TM             1093 non-null   int64  
 9   WD_INS                1093 non-null   int64  
 10  WS_INS                1093 non-null   float64
 11  WS_INS_TM             1093 non-null   int64  
 12  TA_AVG                1093 non-null   float64
 13  TA_MAX                1093 non-null   float64
 14  TA_MAX_TM             1093 non-null   int64  
 15  TA_MIN               